# Task name (owner)

Copy this file to `notebooks/<task>.ipynb`, run the 3 setup cells, then write your code below.

In [ ]:
# 1. Setup (Colab or local)
import os, sys
if not os.path.exists("mgtab.py"):
    if "google.colab" in sys.modules:
        if not os.path.exists("sna-content-moderation"):
            !git clone -q https://github.com/ThSonDev/sna-content-moderation.git
        %cd sna-content-moderation
        !pip install -q -r requirements.txt
    else:
        %cd ..
sys.path.insert(0, os.getcwd())

In [ ]:
# 2. Download MGTAB (31 MB) into data/MGTAB, only once
import urllib.request, zipfile
if not os.path.exists("data/MGTAB/features.pt"):
    os.makedirs("data", exist_ok=True)
    url = "https://drive.usercontent.google.com/download?id=1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe&export=download&confirm=t"
    urllib.request.urlretrieve(url, "data/MGTAB.zip")
    zipfile.ZipFile("data/MGTAB.zip").extractall("data")

In [ ]:
# 3. Load data and the shared split (never make your own split)
import numpy as np
from mgtab import load, relations, split, best_threshold, evaluate, RELATIONS
X, y, stance, E, T, W = load()
rel = relations(E, T, len(y))   # {"follow": (src, dst), ...}
train, val, test = split(y)     # test stays locked until the final run
print(X.shape, {r: len(s) for r, (s, d) in rel.items()})

## Your code

Rules: fit scalers and models on `train` only; report `val` only; tune the threshold with `best_threshold` on `val`;
never use `stance` as a feature. Save result tables to `results/<task>.csv`.

In [ ]:
# Example: LR on all features (delete it and write your own)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(X[train])
clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(sc.transform(X[train]), y[train])
s = clf.predict_proba(sc.transform(X))[:, 1]
evaluate(y[val], s[val], best_threshold(y[val], s[val]))